# SomnoScope — Mejoras para la clasificación de estadios de sueño

Este notebook implementa, sobre los datos del repo `MicroP1MAIA` (Sleep-EDF, 9 sujetos / 18 registros):

1. **Validación leave-one-subject-out (LOSO)**: 9 folds, media ± desviación. El sujeto de test nunca se usa
   para elegir época, pesos del ensamble ni umbrales (en el benchmark original la mejor época se elegía sobre el set de test).
2. **Más señal**: EEG Fpz-Cz + EEG Pz-Oz + EOG horizontal, normalizadas **por registro** (conserva la amplitud relativa, clave para N3).
3. **Contexto temporal en modelos clásicos**: características de las épocas t-2..t+2 + suavizado Viterbi.
4. **Red secuencial** (CNN por época + BiLSTM sobre 20 épocas consecutivas), focal loss, aumentación, AMP en GPU.
5. **Ensamble** LightGBM + red, y **ajuste de sesgos por clase** (para subir N1), ambos calibrados en un sujeto de validación interno.

Ejecuta el notebook desde la **raíz del repo** (donde está la carpeta `data/`).

In [ ]:
# 1. Imports y configuración
import os, time, json, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import lightgbm as lgb
from scipy import signal
from scipy.stats import skew, kurtosis
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

# ---------------- Configuración ----------------
REPO      = Path.cwd()
DATA_DIR  = REPO / "data" / "raw"
CACHE_DIR = REPO / "data" / "processed"
OUT_DIR   = REPO / "reports" / "mejoras"
MODELS_DIR = REPO / "models"

CHANNELS  = ["EEG Fpz-Cz", "EEG Pz-Oz", "EOG horizontal"]
SFREQ, EPOCH_SEC = 100, 30
N_SAMP    = SFREQ * EPOCH_SEC            # 3000 muestras por época
CLASSES   = ["W", "N1", "N2", "N3", "REM"]
SEED      = 42

CTX       = 2        # épocas de contexto a cada lado (modelos clásicos)
SEQ_LEN   = 20       # épocas consecutivas por ventana (red secuencial)

RUN_LGB   = True
RUN_DL    = True
DL_EPOCHS = 40       # pasadas de entrenamiento por fold
DL_BATCH  = 32       # ventanas por batch (cada una = SEQ_LEN épocas)
DL_LR     = 1e-3
LGB_TREES = 400
MAX_RECORDS = None   # p.ej. 6 para una prueba rápida
SAVE_FINAL_MODELS = False   # True: al final entrena y guarda modelos con todos los sujetos

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
else:
    print("⚠ Sin GPU: instala PyTorch con CUDA (ver instrucciones) o el entrenamiento será muy lento.")

def set_seed(s=SEED):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()

## 2. Carga de datos (3 canales, normalización por registro)
Se usa la misma partición de etiquetas AASM (W, N1, N2, N3, REM) y el mismo recorte de vigilia que el repo (60 épocas antes/después del sueño).

In [ ]:
# 2. Carga y preprocesamiento
STAGE_MAP = {"Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
             "Sleep stage 3": 3, "Sleep stage 4": 3, "Sleep stage R": 4}

def find_pairs(data_dir: Path):
    pairs = []
    for psg in sorted(data_dir.rglob("*PSG.edf")):
        prefix = psg.name[:6]                                   # p.ej. SC4001
        hyp = sorted(psg.parent.glob(f"{prefix}*Hypnogram.edf"))
        if hyp:
            pairs.append((psg, hyp[0], prefix[:5], prefix))     # (psg, hipnograma, sujeto, registro)
    return pairs

def load_recording(psg_path: Path, hyp_path: Path):
    raw = mne.io.read_raw_edf(str(psg_path), preload=False, verbose=False)
    missing = [c for c in CHANNELS if c not in raw.ch_names]
    if missing:
        raise ValueError(f"faltan canales {missing} en {psg_path.name}; hay {raw.ch_names}")
    raw.pick(CHANNELS)
    raw.load_data()
    if raw.info["sfreq"] != SFREQ:
        raw.resample(SFREQ)
    raw.filter(0.5, 35.0, fir_design="firwin", verbose=False)
    raw.set_annotations(mne.read_annotations(str(hyp_path)), emit_warning=False)
    events, _ = mne.events_from_annotations(raw, event_id=STAGE_MAP,
                                            chunk_duration=EPOCH_SEC, verbose=False)
    data = raw.get_data()
    starts = events[:, 0] - raw.first_samp
    labels = events[:, 2]
    ok = (starts >= 0) & (starts + N_SAMP <= data.shape[1])
    starts, labels = starts[ok], labels[ok]
    X = np.stack([data[:, s:s + N_SAMP] for s in starts]).astype(np.float32)   # (n, C, 3000)
    y = labels.astype(np.int64)
    # recorte de vigilia excesiva (igual que el repo)
    sleep_idx = np.where(y != 0)[0]
    if len(sleep_idx) > 0:
        a, b = max(0, sleep_idx[0] - 60), min(len(y), sleep_idx[-1] + 60)
        X, y = X[a:b], y[a:b]
    # normalización POR REGISTRO y por canal (conserva amplitud relativa entre épocas)
    mu = X.mean(axis=(0, 2), keepdims=True)
    sd = X.std(axis=(0, 2), keepdims=True) + 1e-12
    X = np.clip((X - mu) / sd, -20, 20).astype(np.float32)
    return X, y

def load_dataset():
    cache = CACHE_DIR / "cache_multich_v2.npz"
    if cache.exists() and MAX_RECORDS is None:
        print("Cargando caché:", cache.name)
        d = np.load(cache)
        return d["X"], d["y"], d["subj"], d["rec"]
    pairs = find_pairs(DATA_DIR)
    if MAX_RECORDS:
        pairs = pairs[:MAX_RECORDS]
    print(f"{len(pairs)} registros encontrados en {DATA_DIR}")
    Xs, ys, subs, recs = [], [], [], []
    for i, (psg, hyp, subj, rec) in enumerate(pairs, 1):
        t0 = time.time()
        try:
            X, y = load_recording(psg, hyp)
        except Exception as e:
            print(f"  [!] {psg.name}: {e}")
            continue
        Xs.append(X); ys.append(y); subs += [subj] * len(y); recs += [rec] * len(y)
        print(f"  [{i}/{len(pairs)}] {rec} ({subj}): {len(y)} épocas, {time.time()-t0:.0f}s, "
              f"dist={np.bincount(y, minlength=5).tolist()}")
    X, y = np.concatenate(Xs), np.concatenate(ys)
    subj, rec = np.array(subs), np.array(recs)
    if MAX_RECORDS is None:
        np.savez_compressed(cache, X=X, y=y, subj=subj, rec=rec)
    return X, y, subj, rec

X, y, subj, rec = load_dataset()
subjects = sorted(np.unique(subj))
print(f"\nX: {X.shape}  | sujetos: {len(subjects)} | registros: {len(np.unique(rec))}")
print("Distribución global [W,N1,N2,N3,REM]:", np.bincount(y, minlength=5).tolist())

# límites (inicio, fin) de cada registro en los arreglos globales (los registros son contiguos)
bounds = {r: (int(np.where(rec == r)[0][0]), int(np.where(rec == r)[0][-1]) + 1) for r in np.unique(rec)}
recs_of = {s: sorted(set(rec[subj == s])) for s in subjects}

## 3. Características espectrales + contexto temporal (modelos clásicos)
Por canal: potencia en bandas (log), potencias relativas, ratios, estadísticos y parámetros de Hjorth.
Luego se estandarizan **por registro** y se apilan las épocas t-2..t+2.

In [ ]:
# 3. Features
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "sigma": (12, 16), "beta": (16, 30)}

def _feat_chunk(x, fs=SFREQ):
    n, c, _ = x.shape
    freqs, psd = signal.welch(x, fs=fs, nperseg=4 * fs, noverlap=2 * fs, axis=-1)
    df = freqs[1] - freqs[0]
    bp = np.stack([psd[..., (freqs >= lo) & (freqs < hi)].sum(-1) * df
                   for lo, hi in BANDS.values()], -1) + 1e-12            # (n,c,5)
    total = bp.sum(-1, keepdims=True)
    rel, logabs, logtot = bp / total, np.log10(bp), np.log10(total)
    d, th, al, si, be = (bp[..., k] for k in range(5))
    ratios = np.stack([np.log10(d / th), np.log10(th / al),
                       np.log10((d + th) / (al + be)), np.log10(si / (al + be))], -1)
    d1 = np.diff(x, axis=-1); d2 = np.diff(d1, axis=-1)
    v0, v1, v2 = x.var(-1) + 1e-12, d1.var(-1) + 1e-12, d2.var(-1) + 1e-12
    mob = np.sqrt(v1 / v0); comp = np.sqrt(v2 / v1) / mob
    stats = np.stack([x.std(-1), skew(x, axis=-1), kurtosis(x, axis=-1), np.ptp(x, axis=-1), mob, comp], -1)
    return np.concatenate([logabs, logtot, rel, ratios, stats], -1).reshape(n, -1).astype(np.float32)

def extract_features(X, chunk=1500):
    return np.concatenate([_feat_chunk(X[i:i + chunk]) for i in range(0, len(X), chunk)])

def standardize_by_record(Fm, rec):
    out = np.empty_like(Fm)
    for r in np.unique(rec):
        m = rec == r
        out[m] = (Fm[m] - Fm[m].mean(0)) / (Fm[m].std(0) + 1e-6)
    return out

def stack_context(Fm, rec, k=CTX):
    idx = np.arange(len(Fm))
    lo = np.array([bounds[r][0] for r in rec]); hi = np.array([bounds[r][1] - 1 for r in rec])
    return np.concatenate([Fm[np.clip(idx + off, lo, hi)] for off in range(-k, k + 1)], axis=1)

t0 = time.time()
F0 = standardize_by_record(extract_features(X), rec)
F_ctx = stack_context(F0, rec, CTX)
print(f"Features por época: {F0.shape[1]} | con contexto ±{CTX}: {F_ctx.shape[1]} ({time.time()-t0:.0f}s)")

## 4. Modelos, métricas y post-procesado

In [ ]:
# 4a. Métricas, LightGBM y utilidades
def metrics(y_true, y_pred):
    f1 = f1_score(y_true, y_pred, average=None, labels=range(5), zero_division=0)
    return dict(acc=accuracy_score(y_true, y_pred), f1_macro=float(f1.mean()),
                kappa=cohen_kappa_score(y_true, y_pred),
                **{f"f1_{c}": float(v) for c, v in zip(CLASSES, f1)})

def f1m(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", labels=range(5), zero_division=0)

def train_lgb(Xtr, ytr, seed=SEED):
    clf = lgb.LGBMClassifier(n_estimators=LGB_TREES, learning_rate=0.05, num_leaves=31,
                             subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                             reg_lambda=1.0, class_weight="balanced",
                             random_state=seed, n_jobs=-1, verbose=-1)
    return clf.fit(Xtr, ytr)

def proba5(clf, Xe):
    p = np.zeros((len(Xe), 5)); p[:, clf.classes_] = clf.predict_proba(Xe); return p

# --- Viterbi (suavizado temporal con matriz de transición aprendida en train) ---
def transition_logmat(y, rec, train_recs):
    T = np.ones((5, 5))
    for r in train_recs:
        yy = y[rec == r]; np.add.at(T, (yy[:-1], yy[1:]), 1)
    return np.log(T / T.sum(1, keepdims=True))

def viterbi(probs, logT, beta):
    n, K = probs.shape
    lp = np.log(probs + 1e-6)
    dp = np.zeros((n, K)); bp = np.zeros((n, K), dtype=int); dp[0] = lp[0]
    for t in range(1, n):
        s = dp[t - 1][:, None] + beta * logT
        bp[t] = s.argmax(0); dp[t] = s.max(0) + lp[t]
    path = np.zeros(n, dtype=int); path[-1] = dp[-1].argmax()
    for t in range(n - 1, 0, -1):
        path[t - 1] = bp[t][path[t]]
    return path

def smooth_records(probs, idx, rec, logT, beta):
    """Aplica Viterbi por registro sobre las posiciones globales `idx` (ordenadas)."""
    pred = probs[idx].argmax(1) if beta == 0 else np.zeros(len(idx), dtype=int)
    if beta == 0:
        return pred
    for r in np.unique(rec[idx]):
        m = rec[idx] == r
        pred[m] = viterbi(probs[idx][m], logT, beta)
    return pred

# --- calibración en el sujeto de validación interno ---
def tune_weight(p_lgb, p_dl, yv):
    best = (-1, 0.5)
    for w in [0, 0.25, 0.5, 0.75, 1.0]:
        f = f1m(yv, (w * p_dl + (1 - w) * p_lgb).argmax(1))
        if f > best[0]: best = (f, w)
    return best[1]

def tune_bias(p, yv, grid=np.linspace(0.5, 2.5, 9), rounds=2):
    m = np.ones(5)
    for _ in range(rounds):
        for c in range(5):
            best = (-1, m[c])
            for g in grid:
                mm = m.copy(); mm[c] = g
                f = f1m(yv, (p * mm).argmax(1))
                if f > best[0]: best = (f, g)
            m[c] = best[1]
    return m

def tune_beta(p, idx_v, yv, rec, logT, betas=(0, 0.25, 0.5, 1.0)):
    scores = [(f1m(yv, smooth_records(p, idx_v, rec, logT, b)), b) for b in betas]
    return max(scores)[1]

## 5. Red secuencial (CNN por época + BiLSTM sobre 20 épocas)
- Entrada: ventanas de `SEQ_LEN` épocas consecutivas × 3 canales × 3000 muestras; salida: una etiqueta **por época**.
- Pérdida focal con pesos por clase (mejora N1), aumentación (escala, ruido, desplazamiento), AdamW + OneCycle, bf16 en GPU.
- Inferencia con ventanas solapadas (promedio ponderado de probabilidades).

In [ ]:
# 5a. Arquitectura
class EpochEncoder(nn.Module):
    def __init__(self, in_ch=3, d=128, p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, 32, 50, 6, 25), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(8, 8), nn.Dropout(p),
            nn.Conv1d(32, 64, 8, 1, 4), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, 64, 8, 1, 4), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(64, 128, 6, 1, 3), nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(4), nn.Dropout(p))
        self.proj = nn.Sequential(nn.Linear(128 * 4, d), nn.GELU())
    def forward(self, x):                       # (B*L, C, T)
        return self.proj(self.net(x).flatten(1))

class SeqSleepNet(nn.Module):
    def __init__(self, in_ch=3, d=128, hidden=128, n_classes=5, p=0.3):
        super().__init__()
        self.enc = EpochEncoder(in_ch, d, p)
        self.lstm = nn.LSTM(d, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=p)
        self.head = nn.Sequential(nn.Dropout(p), nn.Linear(2 * hidden + d, 64), nn.GELU(), nn.Linear(64, n_classes))
    def forward(self, x):                       # (B, L, C, T)
        B, L, C, T = x.shape
        e = self.enc(x.reshape(B * L, C, T)).reshape(B, L, -1)
        h, _ = self.lstm(e)
        return self.head(torch.cat([h, e], -1))  # (B, L, 5)

def focal_loss(logits, target, alpha, gamma=2.0):
    logp = F.log_softmax(logits.float(), -1)
    logpt = logp.gather(1, target[:, None]).squeeze(1)
    return (-alpha[target] * (1 - logpt.exp()) ** gamma * logpt).mean()

def augment(xb):
    B, L, C, T = xb.shape
    scale = torch.empty(B, 1, C, 1, device=xb.device).uniform_(0.8, 1.25)
    xb = xb * scale + 0.05 * torch.randn_like(xb)
    shift = int(np.random.randint(-150, 151))
    return torch.roll(xb, shift, dims=-1) if shift else xb

def amp_ctx():
    return torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=(device.type == "cuda"))

def window_starts(recs, L, stride, offset=0, cover_end=False):
    out = []
    for r in recs:
        lo, hi = bounds[r]
        if hi - lo < L: continue
        s = list(range(lo + offset, hi - L + 1, stride))
        if cover_end and (not s or s[-1] != hi - L): s.append(hi - L)
        out += s
    return np.array(out, dtype=np.int64)

@torch.no_grad()
def predict_seq(model, X_t, recs, L=SEQ_LEN, bs=64):
    """Devuelve (N,5) con probabilidades solo en los registros `recs` (resto en 0)."""
    model.eval()
    N = X_t.shape[0]
    probs = torch.zeros(N, 5, device=X_t.device); wsum = torch.zeros(N, device=X_t.device)
    w = (0.25 + torch.sin(torch.pi * (torch.arange(L, device=X_t.device) + 0.5) / L))
    ar = torch.arange(L, device=X_t.device)
    starts = torch.from_numpy(window_starts(recs, L, max(1, L // 2), cover_end=True)).to(X_t.device)
    for i in range(0, len(starts), bs):
        idx = starts[i:i + bs, None] + ar
        with amp_ctx():
            logits = model(X_t[idx])
        p = F.softmax(logits.float(), -1) * w[None, :, None]
        probs.index_add_(0, idx.reshape(-1), p.reshape(-1, 5))
        wsum.index_add_(0, idx.reshape(-1), w.expand(idx.shape[0], -1).reshape(-1))
    return (probs / wsum.clamp(min=1e-9)[:, None]).cpu().numpy()

def train_seq(X_t, y_t, y_np, train_recs, val_recs=None, epochs=DL_EPOCHS, seed=SEED, verbose=True):
    """Entrena SeqSleepNet. Con val_recs elige la mejor época por F1 macro (sujeto de validación interno);
    sin val_recs entrena `epochs` pasadas fijas."""
    set_seed(seed)
    model = SeqSleepNet(in_ch=X_t.shape[1]).to(device)
    tr_idx = np.concatenate([np.arange(*bounds[r]) for r in train_recs])
    cnt = np.bincount(y_np[tr_idx], minlength=5).astype(np.float64)
    alpha = torch.tensor((1 / np.maximum(cnt, 1)) ** 0.5, dtype=torch.float32, device=device)
    alpha = alpha / alpha.mean()
    stride = SEQ_LEN // 2
    n_win = len(window_starts(train_recs, SEQ_LEN, stride))
    steps = int(np.ceil(n_win / DL_BATCH))
    opt = torch.optim.AdamW(model.parameters(), lr=DL_LR, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=DL_LR, total_steps=epochs * steps, pct_start=0.15)
    ar = torch.arange(SEQ_LEN, device=device)
    if val_recs is not None:
        v_idx = np.concatenate([np.arange(*bounds[r]) for r in val_recs])
    best_f1, best_state, best_ep, hist = -1.0, None, 0, []
    for ep in range(epochs):
        model.train()
        starts = window_starts(train_recs, SEQ_LEN, stride, offset=np.random.randint(stride))
        starts = torch.from_numpy(np.random.permutation(starts)).to(device)
        tot = 0.0
        for i in range(0, len(starts), DL_BATCH):
            idx = starts[i:i + DL_BATCH, None] + ar
            xb, yb = augment(X_t[idx]), y_t[idx]
            with amp_ctx():
                logits = model(xb)
            loss = focal_loss(logits.reshape(-1, 5), yb.reshape(-1), alpha)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item()
        if val_recs is not None:
            pv = predict_seq(model, X_t, val_recs)[v_idx]
            f = f1m(y_np[v_idx], pv.argmax(1))
            hist.append((tot / max(1, steps), f))
            if f > best_f1:
                best_f1, best_ep = f, ep + 1
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            if verbose and ((ep + 1) % 5 == 0 or ep == 0):
                print(f"    ep {ep+1:02d}/{epochs} loss={tot/max(1,steps):.4f} val-F1={f:.4f} (mejor {best_f1:.4f} @ {best_ep})")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, hist

## 6. Validación leave-one-subject-out
Para cada sujeto de test: otro sujeto actúa como **validación interna** (elige época de la red, peso del ensamble,
sesgos por clase y β de Viterbi) y los 7 restantes entrenan.

In [ ]:
# 6. Bucle LOSO
X_t = torch.from_numpy(X).to(device)
y_t = torch.from_numpy(y).to(device)

VARIANTS = ["1. LGB por época (baseline)", "2. LGB + contexto", "3. LGB + contexto + Viterbi",
            "4. Red secuencial", "5. Ensamble LGB-ctx + Red",
            "6. Ensamble + sesgos por clase", "7. Ensamble + sesgos + Viterbi"]
rows, pooled = [], {v: ([], []) for v in VARIANTS}
fold_info = []
t_all = time.time()

for k, test_s in enumerate(subjects):
    t_fold = time.time()
    val_s = subjects[(k + 1) % len(subjects)]
    train_s = [s for s in subjects if s not in (test_s, val_s)]
    tr_recs = [r for s in train_s for r in recs_of[s]]
    va_recs, te_recs = recs_of[val_s], recs_of[test_s]
    tr = np.concatenate([np.arange(*bounds[r]) for r in tr_recs])
    va = np.concatenate([np.arange(*bounds[r]) for r in va_recs])
    te = np.concatenate([np.arange(*bounds[r]) for r in te_recs])
    logT = transition_logmat(y, rec, tr_recs)
    print(f"\n=== Fold {k+1}/{len(subjects)} | test={test_s} | val={val_s} | train={len(train_s)} sujetos ===")

    P = {}                      # probabilidades (N,5) por variante, solo válidas en va y te
    preds = {}

    if RUN_LGB:
        m0 = train_lgb(F0[tr], y[tr]); m1 = train_lgb(F_ctx[tr], y[tr])
        p0 = np.zeros((len(y), 5)); p1 = np.zeros((len(y), 5))
        for ix in (va, te):
            p0[ix] = proba5(m0, F0[ix]); p1[ix] = proba5(m1, F_ctx[ix])
        preds[VARIANTS[0]] = p0[te].argmax(1)
        preds[VARIANTS[1]] = p1[te].argmax(1)
        b1 = tune_beta(p1, va, y[va], rec, logT)
        preds[VARIANTS[2]] = smooth_records(p1, te, rec, logT, b1)

    if RUN_DL:
        model, best_ep, hist = train_seq(X_t, y_t, y, tr_recs, va_recs)
        p2 = np.zeros((len(y), 5))
        pall = predict_seq(model, X_t, va_recs + te_recs)
        p2[va], p2[te] = pall[va], pall[te]
        preds[VARIANTS[3]] = p2[te].argmax(1)
        fold_info.append(dict(test=str(test_s), val=str(val_s), best_epoch=best_ep))
        del model
        if device.type == "cuda": torch.cuda.empty_cache()

    if RUN_LGB and RUN_DL:
        w = tune_weight(p1[va], p2[va], y[va])
        pe = w * p2 + (1 - w) * p1
        preds[VARIANTS[4]] = pe[te].argmax(1)
        bias = tune_bias(pe[va], y[va])
        pb = pe * bias
        preds[VARIANTS[5]] = pb[te].argmax(1)
        b2 = tune_beta(pb, va, y[va], rec, logT)
        preds[VARIANTS[6]] = smooth_records(pb, te, rec, logT, b2)
        fold_info[-1].update(w_dl=w, bias=[round(float(b), 2) for b in bias], beta_lgb=b1, beta_final=b2)

    for v, pr in preds.items():
        m = metrics(y[te], pr); rows.append(dict(fold=k + 1, test=test_s, variante=v, **m))
        pooled[v][0].append(y[te]); pooled[v][1].append(pr)
    last = [v for v in VARIANTS if v in preds][-1]
    print(f"  {last}: F1-macro={rows[-1]['f1_macro']:.3f}  kappa={rows[-1]['kappa']:.3f}  F1-N1={rows[-1]['f1_N1']:.3f}  ({time.time()-t_fold:.0f}s)")
    if fold_info: print("  ", fold_info[-1])

print(f"\nTiempo total: {(time.time()-t_all)/60:.1f} min")

## 7. Resultados

In [ ]:
# 7a. Tabla comparativa
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "resultados_por_fold.csv", index=False)
cols = ["acc", "f1_macro", "kappa", "f1_W", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
mean, std = res.groupby("variante")[cols].mean(), res.groupby("variante")[cols].std()
summary = mean.round(3).astype(str) + " ± " + std.round(3).astype(str)
summary = summary.loc[[v for v in VARIANTS if v in summary.index]]
summary.to_csv(OUT_DIR / "resumen_loso.csv")
try:
    from IPython.display import display
    display(summary)
except Exception:
    print(summary.to_string())
pd.DataFrame(fold_info).to_csv(OUT_DIR / "info_folds.csv", index=False)

In [ ]:
# 7b. Gráfico y matriz de confusión
vs = [v for v in VARIANTS if v in mean.index]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col in zip(axes, ["f1_macro", "f1_N1"]):
    ax.bar(range(len(vs)), mean.loc[vs, col], yerr=std.loc[vs, col], capsize=4, color="#4C78A8")
    ax.set_xticks(range(len(vs))); ax.set_xticklabels([v.split(".")[0] for v in vs])
    ax.set_title(f"{col} (media ± sd entre sujetos, LOSO)"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "comparacion_variantes.png", dpi=150); plt.show()

best_v = max(vs, key=lambda v: mean.loc[v, "f1_macro"])
yt, yp = np.concatenate(pooled[best_v][0]), np.concatenate(pooled[best_v][1])
cm = confusion_matrix(yt, yp, labels=range(5), normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > .5 else "black")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title(f"Matriz de confusión (agregada LOSO)\n{best_v}")
plt.tight_layout(); plt.savefig(OUT_DIR / "matriz_confusion_mejor.png", dpi=150); plt.show()

## 8. (Opcional) Entrenar y guardar los modelos finales con todos los sujetos
Activa `SAVE_FINAL_MODELS = True` en la celda de configuración. La red usa como nº de pasadas la mediana de las mejores épocas de los folds.

In [ ]:
# 8. Modelos finales
if SAVE_FINAL_MODELS:
    import joblib
    all_recs = [r for s in subjects for r in recs_of[s]]
    if RUN_LGB:
        joblib.dump(train_lgb(F_ctx, y), MODELS_DIR / "model_lightgbm_ctx.pkl")
    if RUN_DL:
        ep_final = int(np.median([f["best_epoch"] for f in fold_info]))
        net, _, _ = train_seq(X_t, y_t, y, all_recs, val_recs=None, epochs=ep_final)
        torch.save(net.state_dict(), MODELS_DIR / "model_seqsleepnet.pt")
    print("Modelos guardados en", MODELS_DIR)